In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM mbacp10", con=connection2)

connection2.close()



In [2]:
base2

,acopaccod,acopacdes
0,1,SOLO
1,2,ACOMPANADO


In [3]:
a=base2[base2.duplicated(subset=["acopaccod"])]
a

,acopaccod,acopacdes


In [4]:
base2.columns

Index(['acopaccod', 'acopacdes'], dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_mbacp10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mbacp10 FALSO CON LO OBTENIDO DEL ORACLE

query_count_before = "SELECT COUNT(*) FROM mbacp10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla mbacp10 antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mbacp10 
ALTER COLUMN acopaccod TYPE CHARACTER (1),
ALTER COLUMN acopacdes TYPE CHARACTER (10);


UPDATE mbacp10 
SET 

acopacdes= t.acopacdes


FROM tmp_mbacp10 t 
WHERE mbacp10.acopaccod = t.acopaccod and mbacp10.acopaccod IS NOT NULL and t.acopaccod IS NOT NULL ;

INSERT INTO mbacp10 
(acopaccod, acopacdes) 

SELECT 
acopaccod, acopacdes

FROM tmp_mbacp10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mbacp10 
    WHERE mbacp10.acopaccod = tmp_mbacp10.acopaccod and mbacp10.acopaccod IS NOT NULL and tmp_mbacp10.acopaccod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mbacp10;
DELETE FROM mbacp10 WHERE acopaccod ='';
SELECT COUNT(*) FROM mbacp10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbacp10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

base2 = pd.read_sql_query(f"SELECT * FROM mbacp10", con=connection3)

connection3.close()


Cantidad de filas en la tabla mbacp10 antes de la inserción: 2
Cantidad de filas en la tabla mbacp10 después de la inserción: 2
La cantidad de filas insertadas fue de: 0


In [6]:
base2.columns

Index(['acopaccod', 'acopacdes'], dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_mbacp10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_emeaco"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emeaco antes de la inserción: {cant_antes}")


query="""
ALTER TABLE tmp_mbacp10 
ALTER COLUMN acopaccod TYPE CHARACTER (1),
ALTER COLUMN acopacdes TYPE CHARACTER (10);



INSERT INTO dim_emeaco (cod_aco, des_aco) 
SELECT acopaccod, acopacdes

FROM tmp_mbacp10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_emeaco 
    WHERE dim_emeaco.cod_aco = tmp_mbacp10.acopaccod
);

DROP TABLE tmp_mbacp10;

SELECT COUNT(*) FROM dim_emeaco;
"""

c1 = text(query)
cursor = connection4.execute(c1)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emeaco después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection4.close()

Cantidad de filas en la tabla dim_emeaco antes de la inserción: 2
Cantidad de filas en la tabla dim_emeaco después de la inserción: 2
La cantidad de filas insertadas fue de: 0
